# Notebook 17 — Wyoming Fiscal Baseline Pull

**Purpose:** Assemble the fiscal baseline for all 23 Wyoming counties — the data foundation that Phase W2
(fiscal coefficients) and W3 (engine v2.1) depend on.

**Scope:** Wyoming's 23 counties only. Fiscal institutions are state-specific; assessment ratios,
severance structure, and school finance do not generalise across state lines.

**Design contract:**
- Every value carries `{value, year, source, confidence, notes}` — no bare numbers.
- **Never invent a value.** If a source is unavailable, set `value: null` and log to `MANUAL_FETCH.md`.
- All arithmetic is done by the engine (W3), not here.

**Network strategy (determined at runtime in Cell 5):**
- BLS QCEW API (`data.bls.gov/cew`) and LAUS API (`api.bls.gov`) → live pulls when accessible.
- Wyoming DOR Annual Report (PDF, Google Drive), PILT (form-based), ONRR (JavaScript-required)
  → `null` stubs + `MANUAL_FETCH.md` entries.

**Outputs:**
- `data/processed/wy_county_fiscal_baseline.json`
- `data/processed/wy_fiscal_sources.csv`
- `data/processed/MANUAL_FETCH.md`

In [1]:
import json
import csv
import io
import os
import time
import pathlib
import requests
import pandas as pd
from typing import Optional, Dict, Any

# Paths relative to this notebook's location
NB_DIR = pathlib.Path(os.getcwd())
ROOT = NB_DIR.parent
PROCESSED = ROOT / 'data' / 'processed'

print(f'Root: {ROOT}')
print(f'Processed: {PROCESSED}')
assert PROCESSED.exists(), f'Processed dir missing: {PROCESSED}'

Root: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map
Processed: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed


In [2]:
# ── Wyoming counties: GEOID → name (from mw_study_counties.csv, in_wyoming=True) ──
WY_COUNTIES: Dict[str, str] = {
    '56001': 'Albany',
    '56003': 'Big Horn',
    '56005': 'Campbell',
    '56007': 'Carbon',
    '56009': 'Converse',
    '56011': 'Crook',
    '56013': 'Fremont',
    '56015': 'Goshen',
    '56017': 'Hot Springs',
    '56019': 'Johnson',
    '56021': 'Laramie',
    '56023': 'Lincoln',
    '56025': 'Natrona',
    '56027': 'Niobrara',
    '56029': 'Park',
    '56031': 'Platte',
    '56033': 'Sheridan',
    '56035': 'Sublette',
    '56037': 'Sweetwater',
    '56039': 'Teton',
    '56041': 'Uinta',
    '56043': 'Washakie',
    '56045': 'Weston',
}

assert len(WY_COUNTIES) == 23, f'Expected 23 WY counties, got {len(WY_COUNTIES)}'

# BLS QCEW: pull annual 2023 (most recent complete year)
QCEW_YEAR = 2023

# NAICS industry codes to pull per county
# (code, own_code, label) — own_code 5 = private; 0 = all ownership types
NAICS_TARGETS = [
    ('2121',   5, 'coal_mining_2121'),
    ('2111',   5, 'oil_gas_extraction_2111'),
    ('2211',   5, 'electric_power_2211'),
    ('23',     5, 'construction_23'),
    ('518210', 5, 'data_processing_518210'),
]

print(f'WY counties loaded: {len(WY_COUNTIES)}')
print(f'NAICS targets: {[t[2] for t in NAICS_TARGETS]}')

WY counties loaded: 23
NAICS targets: ['coal_mining_2121', 'oil_gas_extraction_2111', 'electric_power_2211', 'construction_23', 'data_processing_518210']


In [3]:
# ── Value wrapper — every datum in the output carries this envelope ──

def wrap(
    value: Optional[float],
    year: Optional[int],
    source: str,
    confidence: str,
    notes: str = ''
) -> Dict[str, Any]:
    """Canonical value wrapper. confidence: 'high' | 'medium' | 'low'."""
    d: Dict[str, Any] = {
        'value':      value,
        'year':       year,
        'source':     source,
        'confidence': confidence,
    }
    if notes:
        d['notes'] = notes
    return d


def manual_stub(
    fetch_key: str,
    year_hint: Optional[int],
    notes: str = ''
) -> Dict[str, Any]:
    """Stub for values that require manual download (PDF, JS-gated, form-based)."""
    return wrap(
        value=None,
        year=year_hint,
        source=f'MANUAL_FETCH:{fetch_key}',
        confidence='low',
        notes=notes or 'Pending manual data entry — see MANUAL_FETCH.md'
    )


print('Wrapper functions defined.')

Wrapper functions defined.


In [4]:
# ── Network check ──────────────────────────────────────────────────────────────────
# Attempt one BLS QCEW call (Campbell County, 2023 annual).
# If this fails, switch to full-stub mode for BLS data too.

BLS_LIVE = False
DOR_LIVE = False  # DOR Annual Report is PDF-only — never live via API

try:
    _test_url = f'https://data.bls.gov/cew/data/api/{QCEW_YEAR}/a/area/56005.csv'
    _r = requests.get(_test_url, timeout=15)
    if _r.status_code == 200 and 'industry_code' in _r.text:
        BLS_LIVE = True
        print(f'✓ BLS QCEW live — {_r.status_code}, {len(_r.text):,} bytes')
    else:
        print(f'✗ BLS QCEW returned {_r.status_code} or unexpected content')
except Exception as exc:
    print(f'✗ BLS QCEW connection failed: {exc}')

print()
print('DOR Annual Report: PDF only — no live API. All DOR values → MANUAL_FETCH.md stubs.')
print('PILT: form-based interface — no live API. All PILT values → MANUAL_FETCH.md stubs.')
print('ONRR: JavaScript-required — no live API. All ONRR values → MANUAL_FETCH.md stubs.')
print()
print(f'BLS_LIVE = {BLS_LIVE}')

✓ BLS QCEW live — 200, 170,184 bytes

DOR Annual Report: PDF only — no live API. All DOR values → MANUAL_FETCH.md stubs.
PILT: form-based interface — no live API. All PILT values → MANUAL_FETCH.md stubs.
ONRR: JavaScript-required — no live API. All ONRR values → MANUAL_FETCH.md stubs.

BLS_LIVE = True


In [5]:
# ── BLS QCEW: Employment and wages by industry per county ─────────────────────────
#
# URL: https://data.bls.gov/cew/data/api/{year}/a/area/{geoid}.csv
# Returns all ownership × industry × aggregation-level rows for one county-year.
#
# Key columns:
#   area_fips, own_code, industry_code, agglvl_code,
#   annual_avg_emplvl, avg_annual_pay, total_annual_wages, disclosure_code
#
# disclosure_code: '' = not suppressed; 'N' = not disclosable (suppressed cell)
#
# own_code 5 = Private; we use private for all target industries.
# Electric utilities (2211) may have public-sector workers; if own_code 5 is suppressed
# we also check own_code 0 (total).

def fetch_qcew_county(geoid: str, year: int) -> Optional[pd.DataFrame]:
    """Fetch annual QCEW CSV for one county-year. Returns DataFrame or None on error."""
    url = f'https://data.bls.gov/cew/data/api/{year}/a/area/{geoid}.csv'
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        # Normalise string columns that may have surrounding quotes
        for col in ['industry_code', 'own_code', 'disclosure_code']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip().str.strip('"')
        return df
    except Exception as exc:
        print(f'  ERROR fetching {geoid}: {exc}')
        return None


def extract_naics_row(
    df: pd.DataFrame,
    naics: str,
    own: str = '5',
    fallback_own: str = '0'
) -> Dict[str, Any]:
    """
    Pull one (naics, own_code) row from county QCEW data.
    Falls back to own_code 0 (total) if private-sector cell is suppressed.
    Returns a value-wrapped dict for emplvl and avg_annual_pay.
    """
    source_tag = f'BLS QCEW {QCEW_YEAR} NAICS {naics}'

    for own_code in [own, fallback_own]:
        mask = (df['industry_code'] == naics) & (df['own_code'] == own_code)
        rows = df[mask]
        if rows.empty:
            continue
        row = rows.iloc[0]
        disc = str(row.get('disclosure_code', '')).strip()
        if disc in ('N', 'NA'):
            continue  # suppressed; try fallback
        try:
            emplvl = int(float(str(row['annual_avg_emplvl'])))
            pay    = int(float(str(row['avg_annual_pay'])))
            wages  = int(float(str(row['total_annual_wages'])))
            own_label = 'private' if own_code == '5' else f'own_{own_code}'
            return {
                'annual_avg_emplvl': wrap(emplvl, QCEW_YEAR, source_tag, 'high',
                                          f'own_code={own_code} ({own_label})'),
                'avg_annual_pay':    wrap(pay,    QCEW_YEAR, source_tag, 'high',
                                          f'own_code={own_code} ({own_label})'),
                'total_annual_wages':wrap(wages,  QCEW_YEAR, source_tag, 'high',
                                          f'own_code={own_code} ({own_label})'),
            }
        except (ValueError, TypeError):
            continue

    # All attempts exhausted — suppressed or absent
    return {
        'annual_avg_emplvl': wrap(None, QCEW_YEAR, source_tag, 'low',
                                   'Cell suppressed (disclosure_code=N) or industry absent in this county'),
        'avg_annual_pay':    wrap(None, QCEW_YEAR, source_tag, 'low',
                                   'Cell suppressed or absent'),
        'total_annual_wages':wrap(None, QCEW_YEAR, source_tag, 'low',
                                   'Cell suppressed or absent'),
    }


# Pull QCEW for all 23 WY counties
qcew_results: Dict[str, Dict] = {}

if BLS_LIVE:
    print(f'Fetching BLS QCEW {QCEW_YEAR} for {len(WY_COUNTIES)} WY counties...')
    for i, (geoid, county_name) in enumerate(WY_COUNTIES.items()):
        df = fetch_qcew_county(geoid, QCEW_YEAR)
        if df is None:
            qcew_results[geoid] = {}
            continue
        county_employment = {}
        for (naics, own_code, label) in NAICS_TARGETS:
            county_employment[label] = extract_naics_row(df, naics, str(own_code))
        qcew_results[geoid] = county_employment
        print(f'  [{i+1:02d}/23] {county_name} ({geoid}): '
              f'coal={county_employment.get("coal_mining_2121", {}).get("annual_avg_emplvl", {}).get("value", "—")}')
        if i < len(WY_COUNTIES) - 1:
            time.sleep(0.4)  # polite rate limit
    print('\nQCEW pull complete.')
else:
    print('BLS offline — using null stubs for all QCEW employment data.')
    for geoid in WY_COUNTIES:
        qcew_results[geoid] = {
            label: {
                'annual_avg_emplvl': wrap(None, QCEW_YEAR, 'BLS_OFFLINE', 'low', 'BLS QCEW unavailable'),
                'avg_annual_pay':    wrap(None, QCEW_YEAR, 'BLS_OFFLINE', 'low', 'BLS QCEW unavailable'),
                'total_annual_wages':wrap(None, QCEW_YEAR, 'BLS_OFFLINE', 'low', 'BLS QCEW unavailable'),
            }
            for (_, _, label) in NAICS_TARGETS
        }

Fetching BLS QCEW 2023 for 23 WY counties...


  [01/23] Albany (56001): coal=None


  [02/23] Big Horn (56003): coal=None


  [03/23] Campbell (56005): coal=3602


  [04/23] Carbon (56007): coal=None


...

  [19/23] Sweetwater (56037): coal=None


  [20/23] Teton (56039): coal=None


  [21/23] Uinta (56041): coal=None


  [22/23] Washakie (56043): coal=None


  [23/23] Weston (56045): coal=None

QCEW pull complete.


In [6]:
# ── BLS LAUS: Total county labor force ────────────────────────────────────────────
#
# Series format: LAUCN56{county3}0000000006
#   LA = Labor Area | U = not seasonally adjusted | CN = County
#   56 = Wyoming state FIPS
#   {county3} = 3-digit county FIPS (geoid[2:], e.g. '005' for 56005)
#   0000000006 = area code (zeros) + measure 06 (labor force)
#
# API: POST https://api.bls.gov/publicAPI/v1/timeseries/data/
#   v1 API returns M01–M12 (monthly) but NOT M13 (annual average).
#   Annual average is computed here as mean(M01–M12) for most recent complete year.
# CRITICAL: stored as `labor_force_laus` — W4 yields strip uses this exact key.

def build_laus_series_id(geoid: str) -> str:
    county3 = geoid[2:]  # '56005' → '005'
    return f'LAUCN56{county3}0000000006'


def parse_laus_response(data: Dict) -> Dict[str, Optional[tuple]]:
    """
    Compute annual avg labor force from M01–M12 for most recent complete year.
    Returns {seriesID: (value, year) or None}.
    Note: LAUS v1 API does not return M13; we compute the average ourselves.
    """
    out = {}
    for series in data.get('Results', {}).get('series', []):
        sid = series['seriesID']
        # Group monthly values by year
        by_year: Dict[int, list] = {}
        for d in series.get('data', []):
            period = d.get('period', '')
            if not (period.startswith('M') and period != 'M13'):
                continue
            try:
                yr = int(d['year'])
                val = float(d['value'].replace(',', ''))
                by_year.setdefault(yr, []).append(val)
            except (ValueError, KeyError):
                continue
        # Most recent year with all 12 months
        best_yr, best_avg = None, None
        for yr in sorted(by_year.keys(), reverse=True):
            if len(by_year[yr]) == 12:
                best_yr = yr
                best_avg = round(sum(by_year[yr]) / 12)
                break
        out[sid] = (best_avg, best_yr) if best_avg else None
    return out


laus_results: Dict[str, Dict] = {}

if BLS_LIVE:
    series_ids = [build_laus_series_id(geoid) for geoid in WY_COUNTIES]
    print(f'Fetching BLS LAUS for {len(series_ids)} WY counties (M01–M12 avg)...')
    try:
        resp = requests.post(
            'https://api.bls.gov/publicAPI/v1/timeseries/data/',
            json={'seriesid': series_ids},
            headers={'Content-type': 'application/json'},
            timeout=45
        )
        resp.raise_for_status()
        laus_json = resp.json()
        if laus_json.get('status') != 'REQUEST_SUCCEEDED':
            print(f'  LAUS API status: {laus_json.get("status")} — messages: {laus_json.get("message")}')

        parsed = parse_laus_response(laus_json)

        for geoid in WY_COUNTIES:
            sid = build_laus_series_id(geoid)
            result = parsed.get(sid)
            if result is not None:
                val, yr = result
                laus_results[geoid] = wrap(
                    val, yr, f'BLS LAUS {sid}', 'high',
                    f'Annual avg labor force (measure 06); mean of M01-M12 for {yr}'
                )
            else:
                laus_results[geoid] = wrap(
                    None, None, f'BLS LAUS {sid}', 'low',
                    'No complete year (12 months) returned'
                )

        print('LAUS pull complete. Sample values:')
        for geoid in ['56005', '56021', '56039']:  # Campbell, Laramie, Teton
            v = laus_results.get(geoid, {})
            print(f'  {WY_COUNTIES[geoid]} ({geoid}): labor_force = {v.get("value")} ({v.get("year")})')

    except Exception as exc:
        print(f'LAUS pull failed: {exc}')
        for geoid in WY_COUNTIES:
            sid = build_laus_series_id(geoid)
            laus_results[geoid] = wrap(None, None, f'BLS LAUS {sid}', 'low', f'Fetch error: {exc}')
else:
    print('BLS offline — using null stubs for LAUS labor force.')
    for geoid in WY_COUNTIES:
        sid = build_laus_series_id(geoid)
        laus_results[geoid] = wrap(None, None, f'BLS LAUS {sid}', 'low', 'BLS LAUS unavailable')


Fetching BLS LAUS for 23 WY counties (M01–M12 avg)...


LAUS pull complete. Sample values:
  Campbell (56005): labor_force = 24122 (2024)
  Laramie (56021): labor_force = 48398 (2024)
  Teton (56039): labor_force = 16644 (2024)


## Data Sources — Loaded from Extracted CSVs

Most fiscal data is now populated from CSV files in
`data/processed/WYO DOR report tables/`. Cell 8 loads all available data;
remaining nulls are **genuine data gaps** (county-level breakdown not in the
extracted files), not stubs.

| Item | Source CSV | Status |
|------|-----------|--------|
| Assessed values: mineral (state-assessed) | `state_assessed_values.csv` row 14 | ✓ Loaded |
| Assessed values: local (industrial, commercial, residential, agricultural) | `locally_assessed_values.csv` rows 7/11/15/47 | ✓ Loaded |
| Assessed values: all_other (state utilities) | `state_assessed_values.csv` row 13 | ✓ Loaded |
| Composite mill levy per county | `county_statewide_weighted_average_mill_levies.csv` | ✓ Loaded |
| Mineral production (total, all commodities) | `total_mineral_taxable_value_by_county.csv` | ✓ Loaded |
| Mineral production (coal/oil/gas/trona per county) | `mineral_severance_tax_distribution.csv` | ⚠ Statewide only → null per county |
| Severance tax distributions | `mineral_severance_tax_distribution.csv` | ⚠ Formula-applied pro-rata (confidence: low) |
| PILT payments | `property_tax_distributions_by_county.csv` | ✓ Loaded |
| Sales tax distributions | `sales_use_tax_distribution_report.csv` row 0 | ✓ Loaded (sales tax; use tax has alignment issue) |
| Federal mineral royalties (ONRR) | `fiscal_year_disbursements_wyoming.csv` | ⚠ State-level only (County column is NaN throughout) |
| School finance foundation net | N/A | ⚠ Pending MANUAL_FETCH.md |


In [7]:
# ── Load DOR/PILT data from extracted CSVs ──────────────────────────────────────
#
# Source directory: data/raw/wy_fiscal/extracted/
# Data year: 2025 (DOR 2025 Annual Report, assessment year 2025)
#
# File → fiscal field mapping:
#   Locally Assessed Values .csv                             → assessed_value by property class
#   State Assessed Values.csv                                → state_assessed additions per county
#   Comparison of State and Local Assessed Values.csv        → total_assessed (sanity denominator)
#   County Taxes Levied.csv                                  → mill levy components (reference)
#   Grand total all taxes levied.csv                         → composite_mill_levy (primary W2 input)
#   County and Statewide weighted average mill levies.csv    → mineral_weighted_mill_levy
#   mineral severance tax distribution.csv                   → state_severance_by_mineral (top-level)
#   Sales and use tax distribution report.csv                → sales_use_tax_distribution
#   Percentage of local and state assessed values.csv        → state_class_shares (top-level)
#   pdf_print_counties.cfm.csv                               → pilt_payments
#   Total mineral taxable value of production by county.csv  → mineral_production_valuation
#   table of distributions by county.csv                     → SKIP (unrecoverable raster image)

import re
import numpy as np

EXTRACTED = ROOT / 'data' / 'raw' / 'wy_fiscal' / 'extracted'
DATA_YEAR = 2025  # DOR 2025 Annual Report assessment year

assert EXTRACTED.exists(), f'extracted/ dir missing: {EXTRACTED}'

# ── Print column headers of every CSV in extracted/ ──────────────────────────────
print('=' * 70)
print('CSV COLUMN HEADERS — extracted/')
print('=' * 70)
for csv_path in sorted(EXTRACTED.glob('*.csv')):
    df_peek = pd.read_csv(csv_path, nrows=0)
    print(f'\n{csv_path.name}:')
    for i, col in enumerate(df_peek.columns):
        print(f'  [{i:2d}] {col}')
print()
print('=' * 70)
print()

# ── Dollar/number parser ──────────────────────────────────────────────────────────
def parse_dollar(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    s = re.sub(r'[\$,\s\xa0]', '', str(s).strip())
    if s in ('-', '', 'nan', 'NaN', 'None', 'null'):
        return None
    try:
        return float(s)
    except ValueError:
        return None

# ── Load all CSVs ─────────────────────────────────────────────────────────────────
df_local   = pd.read_csv(EXTRACTED / 'Locally Assessed Values .csv')
df_state   = pd.read_csv(EXTRACTED / 'State Assessed Values.csv')
df_compare = pd.read_csv(EXTRACTED / 'Comparison of State and Local Assessed Values.csv')
df_taxes   = pd.read_csv(EXTRACTED / 'County Taxes Levied.csv')
df_grand   = pd.read_csv(EXTRACTED / 'Grand total all taxes levied.csv')
df_mills   = pd.read_csv(EXTRACTED / 'County and Statewide weighted average mill levies.csv')
df_sev     = pd.read_csv(EXTRACTED / 'mineral severance tax distribution.csv')
df_sut     = pd.read_csv(EXTRACTED / 'Sales and use tax distribution report.csv')
df_pct     = pd.read_csv(EXTRACTED / 'Percentage of local and state asssessed values.csv')
df_pilt    = pd.read_csv(EXTRACTED / 'pdf_print_counties.cfm.csv')
df_mineral = pd.read_csv(EXTRACTED / 'Total mineral taxable value of production by county.csv')
# table of distributions by county.csv → SKIP (unrecoverable raster image; see MANUAL_FETCH.md)
print('All CSVs loaded.')

# ── Source citation strings ───────────────────────────────────────────────────────
SRC_LOCAL   = 'Locally Assessed Values .csv (WY DOR 2025 Annual Report)'
SRC_STATE   = 'State Assessed Values.csv (WY DOR 2025 Annual Report)'
SRC_COMPARE = 'Comparison of State and Local Assessed Values.csv (WY DOR 2025)'
SRC_GRAND   = 'Grand total all taxes levied.csv (WY DOR 2025)'
SRC_MILLS   = 'County and Statewide weighted average mill levies.csv (WY DOR 2025)'
SRC_SEV     = 'mineral severance tax distribution.csv (WY DOR 2025)'
SRC_SUT     = 'Sales and use tax distribution report.csv (WY DOR 2025)'
SRC_PILT    = 'pdf_print_counties.cfm.csv (DOI PILT)'
SRC_MINERAL = 'Total mineral taxable value of production by county.csv (WY DOR 2025)'

name_to_geoid = {name: geoid for geoid, name in WY_COUNTIES.items()}
SKIP_ROWS = {'Totals', 'Grand Total', 'Statewide', 'Total', 'nan', 'None', ''}

# ── 1. Locally Assessed Values ────────────────────────────────────────────────────
# Key columns (verified via Weston spot-check — sum of 6 classes = total_assessed_sanity):
#   [12] Personal Improvements Total Land, Commercial Property & → commercial
#   [16] Personal Improvements Total Land, Residential Property & → residential
#   [48] Total Property Industrial                               → industrial
#   [49] Total Land Agricultural Valuation                      → agricultural
#   [53] Total Locally Assessed                                 → local total (sanity)
local_by_name = {}
for _, row in df_local.iterrows():
    name = str(row['County']).strip()
    if name in SKIP_ROWS:
        continue
    local_by_name[name] = row
assert len(local_by_name) == 23, f'Expected 23 local assessed rows, got {len(local_by_name)}'

# ── 2. State Assessed Values ──────────────────────────────────────────────────────
# Key columns:
#   Minerals      → mineral assessed value (state-assessed at 100% FMV)
#   Non-Minerals  → all_other (electric utilities, pipelines, rail, telecom, airline)
state_by_name = {}
for _, row in df_state.iterrows():
    name = str(row['County']).strip()
    if name in SKIP_ROWS:
        continue
    state_by_name[name] = row
assert len(state_by_name) == 23, f'Expected 23 state assessed rows, got {len(state_by_name)}'

# ── 3. Comparison CSV — total_assessed sanity denominator ────────────────────────
# metric_6 = state_assessed + local_assessed = grand total assessed.
# Verified: Weston state(84,130,711) + local(88,307,957) = metric_6(172,438,668) ✓
compare_by_name = {}
for _, row in df_compare.iterrows():
    name = str(row['County']).strip()
    if name in SKIP_ROWS:
        continue
    compare_by_name[name] = row
assert len(compare_by_name) == 23, f'Expected 23 comparison rows, got {len(compare_by_name)}'

# ── 4. Grand total all taxes levied — composite_mill_levy (primary W2 input) ─────
# Column '_1' = total composite mill levy (mills), all levying entities combined.
grand_by_name = {}
for _, row in df_grand.iterrows():
    name = str(row['County']).strip()
    if name in SKIP_ROWS:
        continue
    grand_by_name[name] = row
assert len(grand_by_name) == 23, f'Expected 23 grand total rows, got {len(grand_by_name)}'

# ── 5. County and Statewide weighted average mill levies → mineral_weighted_mill ──
# Col 0 = county name | col 'nan' = mill levy | col 'nan_2' = production_tax_assessed
mills_name_col = df_mills.columns[0]
mills_by_name = {}
for _, row in df_mills.iterrows():
    name = str(row[mills_name_col]).strip()
    if name in SKIP_ROWS or name == 'Statewide':
        continue
    mills_by_name[name] = {
        'mill_levy':               parse_dollar(row['nan']),
        'production_tax_assessed': parse_dollar(row['nan_2']),
    }
assert len(mills_by_name) == 23, f'Expected 23 mill levy rows, got {len(mills_by_name)}'

# ── 6. Total mineral taxable value by county ─────────────────────────────────────
# "Percentage of Statewide Total" is in percent (0.196 = 0.196%); divide by 100 for fraction.
mineral_by_name = {}
statewide_mineral_total = None
for _, row in df_mineral.iterrows():
    name = str(row['County']).strip()
    if name == 'Grand Total':
        statewide_mineral_total = parse_dollar(row['Total Mineral Taxable Valuation'])
        continue
    if name in SKIP_ROWS:
        continue
    pct_raw = parse_dollar(row['Percentage of Statewide Total'])
    mineral_by_name[name] = {
        'total':        parse_dollar(row['Total Mineral Taxable Valuation']),
        'pct_fraction': (pct_raw / 100.0) if pct_raw is not None else None,
    }
assert statewide_mineral_total is not None, 'Grand Total row not found in mineral CSV'
assert len(mineral_by_name) == 23, f'Expected 23 mineral rows, got {len(mineral_by_name)}'
print(f'Statewide mineral taxable total: ${statewide_mineral_total:,.0f}')

# ── 7. PILT — pdf_print_counties.cfm.csv ─────────────────────────────────────────
# Columns: LOCAL UNIT OF GOVERNMENT | TOTAL PD THIS FY | TOTAL ACRES
pilt_by_name = {}
for _, row in df_pilt.iterrows():
    name = str(row['LOCAL UNIT OF GOVERNMENT']).strip()
    if name in SKIP_ROWS or name == 'Total':
        continue
    pilt_by_name[name] = parse_dollar(row['TOTAL PD THIS FY'])
assert len(pilt_by_name) == 23, f'Expected 23 PILT rows, got {len(pilt_by_name)}'

# ── 8. Sales and use tax distribution ────────────────────────────────────────────
# Operative figure: grand_total for all counties.
# Park (56029) and Sublette (56035): county_sales_tax = $0 is confirmed genuine.
#   These counties do not levy county-option sales/use tax.
#   grand_total reflects state and other allocations only.
NO_COUNTY_SALES_TAX = {'56029', '56035'}

sut_by_name = {}
for _, row in df_sut.iterrows():
    name = str(row['County']).strip()
    if name in SKIP_ROWS:
        continue
    sut_by_name[name] = {
        'county_sales_tax': parse_dollar(row['county_sales_tax']),
        'county_use_tax':   parse_dollar(row['county_use_tax']),
        'county_total':     parse_dollar(row['county_total']),
        'grand_total':      parse_dollar(row['grand_total']),
    }
assert len(sut_by_name) == 23, f'Expected 23 sales/use rows, got {len(sut_by_name)}'

# ── 9. STATE_SEVERANCE_BY_MINERAL — top-level, NOT per county ────────────────────
# Row index 4 = statewide severance tax distribution totals by mineral type.
# Totals column = $774,028,998 (verified against MANUAL_FETCH.md prior estimate).
sev_row = df_sev.iloc[4]
sev_total = parse_dollar(str(sev_row['Totals']))
assert sev_total is not None and sev_total > 7e8, f'Unexpected sev_total: {sev_total}'

MINERAL_SEV_COLS = {
    'coal_surface':     'Coal (Surface)',
    'coal_underground': 'Coal (Underground)',
    'oil':              'Oil #',
    'natural_gas':      'Natural Gas #',
    'trona':            'Trona',
    'uranium':          'Uranium*',
    'sand_gravel':      'Sand & Gravel',
    'bentonite':        'Bentonite',
    'gypsum':           'Gypsum',
    'limestone':        'Limestone',
    'frac_sand':        'Frac Sand',
    'granite_ballast':  'Granite Ballast',
    'shale':            'Shale',
    'moss_rock':        'Moss Rock',
    'decorative_stone': 'Decorative Stone',
    'clay':             'Clay',
    'jade':             'Jade',
    'leonardite':       'Leonardite',
    'gold':             'Gold**',
}

STATE_SEVERANCE_BY_MINERAL = {
    'source':     SRC_SEV,
    'year':       DATA_YEAR,
    'confidence': 'high',
    'note': (
        'Statewide severance tax distribution totals by mineral type (row 4 of CSV). '
        'Not available at county level in this CSV. '
        'Used at top level of wy_county_fiscal_baseline.json for W2 coefficient derivation. '
        'Per-county severance estimated pro-rata in county records (confidence: low).'
    ),
    'total_usd': wrap(sev_total, DATA_YEAR, SRC_SEV, 'high',
                      'Row 4 Totals column of mineral severance tax distribution.csv'),
    'by_mineral': {},
}
for key, col in MINERAL_SEV_COLS.items():
    if col in df_sev.columns:
        val = parse_dollar(str(sev_row[col]))
        STATE_SEVERANCE_BY_MINERAL['by_mineral'][key] = wrap(
            val, DATA_YEAR, SRC_SEV, 'high',
            f'Row 4, column "{col}" of mineral severance tax distribution.csv'
        )

print(f'STATE_SEVERANCE_BY_MINERAL total: ${sev_total:,.0f}')
print(f'  coal_surface: ${STATE_SEVERANCE_BY_MINERAL["by_mineral"]["coal_surface"]["value"]:,.0f}')
print(f'  oil:          ${STATE_SEVERANCE_BY_MINERAL["by_mineral"]["oil"]["value"]:,.0f}')
print(f'  natural_gas:  ${STATE_SEVERANCE_BY_MINERAL["by_mineral"]["natural_gas"]["value"]:,.0f}')
print(f'  trona:        ${STATE_SEVERANCE_BY_MINERAL["by_mineral"]["trona"]["value"]:,.0f}')

# ── 10. STATE_CLASS_SHARES — top-level, NOT per county ───────────────────────────
# Column names garbled by PDF extraction (cid: Unicode escape sequences).
# Stored raw at top level for W2 coefficient derivation.
STATE_CLASS_SHARES = {
    'source':     'Percentage of local and state asssessed values.csv (WY DOR 2025)',
    'year':       DATA_YEAR,
    'confidence': 'medium',
    'note': (
        'Statewide percentage breakdown of assessed value by property class and utility type. '
        'Column names garbled by PDF extraction (cid: Unicode sequences). '
        'Row 1 contains the assessed values by class. '
        'Not used for county rows. Stored at top level for W2 coefficient derivation.'
    ),
    'raw_assessed_values': (df_pct.iloc[1].dropna().to_dict() if len(df_pct) > 1 else {}),
}
print(f'STATE_CLASS_SHARES stored ({len(df_pct)} rows in source CSV).')

# ── Build all county fiscal dicts keyed by GEOID ─────────────────────────────────
DOR_ASSESSED     = {}
DOR_MILL_LEVY    = {}
DOR_MINERAL_PROD = {}
DOR_SEVERANCE    = {}
ONRR_FMR         = {}
DOI_PILT         = {}
DOR_SALES_USE    = {}
gaps = []

def _gap(county, field, src):
    gaps.append((county, field))
    return wrap(None, DATA_YEAR, src, 'low',
                f'Gap: {county} not found or null in {src}')

for geoid, cn in WY_COUNTIES.items():

    s_row = state_by_name.get(cn)
    l_row = local_by_name.get(cn)
    c_row = compare_by_name.get(cn)
    g_row = grand_by_name.get(cn)

    # ── Assessed values ──────────────────────────────────────────────────────────
    mineral_av     = parse_dollar(s_row['Minerals'])     if s_row is not None else None
    all_other_av   = parse_dollar(s_row['Non-Minerals']) if s_row is not None else None
    industrial_av  = parse_dollar(l_row['Total Property Industrial'])                              if l_row is not None else None
    commercial_av  = parse_dollar(l_row['Personal Improvements Total Land, Commercial Property &']) if l_row is not None else None
    residential_av = parse_dollar(l_row['Personal Improvements Total Land, Residential Property &'])if l_row is not None else None
    agricultural_av= parse_dollar(l_row['Total Land Agricultural Valuation'])                       if l_row is not None else None
    total_assessed  = parse_dollar(c_row['metric_6'])    if c_row is not None else None

    def _av(val, col_name, src):
        if val is None:
            return _gap(cn, 'assessed_value.' + col_name, src)
        return wrap(val, DATA_YEAR, src, 'high', 'Column: ' + col_name)

    DOR_ASSESSED[geoid] = {
        'mineral':      _av(mineral_av,     'Minerals',      SRC_STATE),
        'all_other':    _av(all_other_av,   'Non-Minerals',  SRC_STATE),
        'industrial':   _av(industrial_av,  'Total Property Industrial', SRC_LOCAL),
        'commercial':   _av(commercial_av,
                            'Personal Improvements Total Land, Commercial Property &', SRC_LOCAL),
        'residential':  _av(residential_av,
                            'Personal Improvements Total Land, Residential Property &', SRC_LOCAL),
        'agricultural': _av(agricultural_av,'Total Land Agricultural Valuation', SRC_LOCAL),
        'total_assessed_sanity': wrap(total_assessed, DATA_YEAR, SRC_COMPARE, 'high',
            'Grand total assessed (state+local) from Comparison CSV metric_6. '
            'Sanity denominator: equals sum of the 6 class fields above.'),
    }

    # ── Composite mill levy — Grand total all taxes levied.csv col _1 ────────────
    mill_val = parse_dollar(g_row['_1']) if g_row is not None else None
    if mill_val is None:
        gaps.append((cn, 'composite_mill_levy'))
    DOR_MILL_LEVY[geoid] = wrap(
        mill_val, DATA_YEAR, SRC_GRAND, 'high' if mill_val is not None else 'low',
        'Composite mill levy (mills). Column _1 of Grand total all taxes levied.csv. '
        'Includes all levying entities (county + school district + special districts). '
        'Primary W2 input for property tax revenue coefficient.'
    )

    # ── Mineral production valuation ─────────────────────────────────────────────
    m_entry  = mineral_by_name.get(cn, {})
    m_total  = m_entry.get('total')
    m_pct    = m_entry.get('pct_fraction')
    if m_total is None:
        gaps.append((cn, 'mineral_production.total'))

    mw = mills_by_name.get(cn, {})
    mw_mill = mw.get('mill_levy')
    mw_prod = mw.get('production_tax_assessed')

    sev_est   = (m_pct * sev_total) if (m_pct is not None and sev_total) else None
    share_str = f'{m_pct:.5f}' if m_pct is not None else 'N/A'

    DOR_MINERAL_PROD[geoid] = {
        'total_all_commodities': wrap(m_total, DATA_YEAR, SRC_MINERAL, 'high',
            'Total mineral production taxable value (all commodities: coal, oil, gas, trona, other). '
            'Total mineral taxable value of production by county.csv.'),
        'pct_statewide': wrap(m_pct, DATA_YEAR, SRC_MINERAL, 'high',
            'County share of WY statewide mineral taxable valuation (fraction). '
            '"Percentage of Statewide Total" column divided by 100.'),
        'mineral_weighted_mill_levy': wrap(mw_mill, DATA_YEAR, SRC_MILLS, 'high',
            'Production-tax-weighted average mill levy (mills). '
            'County and Statewide weighted average mill levies.csv col "nan".'),
        'production_tax_assessed': wrap(mw_prod, DATA_YEAR, SRC_MILLS, 'high',
            'Total ad valorem production tax assessed value (USD). '
            'County and Statewide weighted average mill levies.csv col "nan_2".'),
        'coal':  wrap(None, DATA_YEAR, 'MANUAL_FETCH:mineral_prod_coal_by_county',  'low',
                      'Per-county coal valuation not in extracted CSVs. '
                      'See MANUAL_FETCH.md Item 1 (DOR Mineral Valuation Report).'),
        'oil':   wrap(None, DATA_YEAR, 'MANUAL_FETCH:mineral_prod_oil_by_county',   'low',
                      'Per-county oil valuation not in extracted CSVs. See MANUAL_FETCH.md Item 1.'),
        'gas':   wrap(None, DATA_YEAR, 'MANUAL_FETCH:mineral_prod_gas_by_county',   'low',
                      'Per-county gas valuation not in extracted CSVs. See MANUAL_FETCH.md Item 1.'),
        'trona': wrap(None, DATA_YEAR, 'MANUAL_FETCH:mineral_prod_trona_by_county', 'low',
                      'Per-county trona valuation not in extracted CSVs. See MANUAL_FETCH.md Item 1.'),
    }

    # ── Severance tax (pro-rata estimate; confidence: low) ───────────────────────
    DOR_SEVERANCE[geoid] = wrap(
        sev_est, DATA_YEAR, 'formula_applied', 'low',
        'Estimated: mineral share ' + share_str + ' x statewide severance $' + f'{sev_total:,.0f}. '
        'Crude pro-rata assuming uniform effective rate across all commodities. '
        'Replace with actual per-county distribution (MANUAL_FETCH.md Item 3).'
    )

    # ── ONRR (county level still unavailable) ────────────────────────────────────
    ONRR_FMR[geoid] = wrap(
        None, DATA_YEAR, 'MANUAL_FETCH:ONRR_county_disbursements', 'low',
        'County-level ONRR disbursements not in extracted CSVs. '
        'See MANUAL_FETCH.md Item 2: revenuedata.onrr.gov/downloads/disbursements/'
    )

    # ── PILT — pdf_print_counties.cfm.csv ─────────────────────────────────────────
    pilt_val = pilt_by_name.get(cn)
    if pilt_val is None:
        gaps.append((cn, 'pilt'))
    DOI_PILT[geoid] = wrap(
        pilt_val, DATA_YEAR, SRC_PILT, 'high' if pilt_val is not None else 'low',
        'TOTAL PD THIS FY from pdf_print_counties.cfm.csv (DOI PILT FY 2025).'
    )

    # ── Sales & use tax — grand_total as operative figure ────────────────────────
    sut = sut_by_name.get(cn, {})
    county_st = sut.get('county_sales_tax')
    grand_tot = sut.get('grand_total')

    if grand_tot is not None and grand_tot > 0:
        if geoid in NO_COUNTY_SALES_TAX:
            note = (
                'County does not levy county-option sales/use tax; '
                'grand_total reflects state and other allocations only. '
                'county_sales_tax = $0 is confirmed genuine (not a data error).'
            )
        else:
            note = (
                'grand_total from Sales and use tax distribution report.csv. '
                'Includes county, municipal, state lodging, and general purpose components. '
                'county_sales_tax component: $' + f'{county_st:,.2f}.'
            )
        DOR_SALES_USE[geoid] = wrap(grand_tot, DATA_YEAR, SRC_SUT, 'high', note)
    else:
        gaps.append((cn, 'sales_use_tax'))
        DOR_SALES_USE[geoid] = wrap(
            None, DATA_YEAR, SRC_SUT, 'low',
            'Gap: ' + cn + ' has null or zero grand_total in sales/use tax CSV.'
        )

# ── Verification ─────────────────────────────────────────────────────────────────
print(f'\nBuilt fiscal dicts for {len(DOR_ASSESSED)} counties.')
if gaps:
    print(f'\n[WARNING] GAPS ({len(gaps)}):')
    for item, reason in gaps:
        print(f'  {item}: {reason}')
else:
    print('No gaps — all 23 counties found in all source CSVs. [OK]')

print('\nSample — Campbell County (56005):')
av = DOR_ASSESSED['56005']
cls_keys = ['mineral', 'all_other', 'industrial', 'commercial', 'residential', 'agricultural']
vals = [av[k]['value'] for k in cls_keys if av[k]['value'] is not None]
total_av = sum(vals)
for k in cls_keys:
    v = av[k]['value']
    pct = '(' + f'{v/total_av:.1%}' + ')' if v else ''
    print(f'  {k:<14}: ${v:>15,.0f}  {pct}' if v else f'  {k:<14}: None')
print(f'  {"TOTAL":<14}: ${total_av:>15,.0f}')
sanity_v = av["total_assessed_sanity"]["value"]
print(f'  sanity_total : ${sanity_v:>15,.0f}  (match: {abs(total_av - sanity_v) < 1})')
print(f'  mill_levy    : {DOR_MILL_LEVY["56005"]["value"]} mills (composite)')
mp = DOR_MINERAL_PROD['56005']
print(f'  mineral_total: ${mp["total_all_commodities"]["value"]:,.0f}')
print(f'  min_wt_mill  : {mp["mineral_weighted_mill_levy"]["value"]} mills')
print(f'  pilt         : ${DOI_PILT["56005"]["value"]:,.0f}')
print(f'  sales_tax    : ${DOR_SALES_USE["56005"]["value"]:,.0f}  (grand_total)')
sev_val = DOR_SEVERANCE['56005']['value']
print(f'  sev_est      : ${sev_val:,.0f}' if sev_val else '  sev_est      : None')


CSV COLUMN HEADERS — extracted/

Comparison of State and Local Assessed Values.csv:
  [ 0] County
  [ 1] metric_0
  [ 2] metric_1
  [ 3] metric_2
  [ 4] metric_3
  [ 5] metric_4
  [ 6] metric_5
  [ 7] metric_6
  [ 8] metric_7
  [ 9] metric_8

County Taxes Levied.csv:
  [ 0] County
  [ 1] Mills
  [ 2] Amount
  [ 3] Mills_1
  [ 4] Amount_1
  [ 5] Mills_2
  [ 6] Amount_2
  [ 7] Mills_3
  [ 8] Amount_3
  [ 9] Mills_4
  [10] Amount_4
  [11] Mills_5
  [12] Amount_5
  [13] Mills_6
  [14] Amount_6
  [15] Mills_7
  [16] Amount_7
  [17] Mills_8
  [18] Amount_8
  [19] Mills_9
  [20] Amount_9
  [21] -
  [22] -_1
  [23] -_2
  [24] -_3
  [25] -_4
  [26] -_5
  [27] 12.000
  [28] 8,017,775
  [29] -_6
  [30] -_7
  [31] 12.000_1
  [32] 8,017,775_1

County and Statewide weighted average mill levies.csv:
  [ 0] 2025 Average Mill County Levies Total Ad Valorem Production Tax Assessed
  [ 1] nan
  [ 2] nan_2

Grand total all taxes levied.csv:
  [ 0] County
  [ 1] nan
  [ 2] 17.232%
  [ 3] 2.039%
  [ 4] 8.53

In [8]:
# ── Assemble wy_county_fiscal_baseline ────────────────────────────────────────────
#
# One record per WY county. Every value uses the canonical wrapper.
# BLS QCEW and LAUS data are live (when BLS is accessible).
# All DOR/PILT/ONRR values are null stubs pending MANUAL_FETCH.md completion.

# Data center equipment exemption — structural field, same for all WY counties.
# (The exemption applies statewide; whether a county has a data center is a separate question.)
DATACENTER_EXEMPTION = {
    'statute':       'W.S. 39-15-105(a)(viii)(O)',
    'description':   'Qualifying data center equipment and infrastructure exempt from sales/use tax.',
    'applies':       True,
    'exempt_share_of_construction_sales_tax': wrap(
        None, None, 'structural_field', 'low',
        'No published county-level figure for exempt share. '
        'W2 will carry this as confidence:low in coefficient derivation. '
        'The exemption is structural: megawatts arrive; part of tax base does not.'
    )
}

fiscal_baseline: Dict[str, Dict] = {}

for geoid, county_name in WY_COUNTIES.items():
    fiscal_baseline[geoid] = {
        'geoid':                geoid,
        'county_name':          county_name,
        'state':                'WY',
        'fiscal_schema_version': '1.0',

        # ── Item 1: Assessed values by property class ──────────────────────────
        # Post-ratio assessed values (not FMV).
        # Assessment ratios (W.S. 39-13-103): minerals 100%, industrial 11.5%, all other 9.5%.
        # FMV of new builds uses capex from action library v3.1 (W2 task).
        'assessed_values': DOR_ASSESSED[geoid],

        # ── Item 2: Composite mill levy ────────────────────────────────────────
        # Unit: mills. Property tax = (assessed_value / 1000) × mill_levy.
        # Statutory components: school foundation (12), local school (25),
        #   county general (~12), special districts (variable).
        'mill_levy_composite_mills': DOR_MILL_LEVY[geoid],

        # ── Item 3: Ad valorem mineral production valuation ────────────────────
        # Production-linked tax base — separate ledger from real property.
        # Coal retirement hits this directly; new energy assets do not.
        'mineral_production_valuation': DOR_MINERAL_PROD[geoid],

        # ── Item 4: Severance tax and federal mineral royalties ────────────────
        # Two distinct streams on different ledgers with different lags.
        'severance_tax_distribution_usd': DOR_SEVERANCE[geoid],
        'federal_mineral_royalties_usd':  ONRR_FMR[geoid],

        # ── Item 5: PILT ───────────────────────────────────────────────────────
        'pilt_usd': DOI_PILT[geoid],

        # ── Item 6: Sales & use tax + data center exemption ───────────────────
        'sales_use_tax_distribution_usd':     DOR_SALES_USE[geoid],
        'datacenter_equipment_exemption':     DATACENTER_EXEMPTION,

        # ── Item 7: Employment and wages by NAICS (BLS QCEW) ──────────────────
        # Fields: annual_avg_emplvl, avg_annual_pay, total_annual_wages per industry.
        # Suppressed cells (disclosure_code='N') carry value:null, confidence:low.
        'employment_bls_qcew': {
            'data_year': QCEW_YEAR,
            'naics_industries': qcew_results.get(geoid, {}),
        },

        # ── Labor force total (BLS LAUS) ───────────────────────────────────────
        # CRITICAL: field name 'labor_force_laus' — W4 yields strip uses this key.
        'labor_force_laus': laus_results.get(geoid, wrap(
            None, None, 'BLS LAUS not fetched', 'low', ''
        )),

        # ── School finance note ────────────────────────────────────────────────
        # Foundation program guarantee partially decouples local school revenue
        # from local assessed valuation. Campbell County exports revenue via recapture
        # (high mineral AV → above-guarantee receipts recaptured).
        # W2 will model foundation net flow per county as confidence:low.
        'school_finance_foundation_net_usd': wrap(
            None, 2023, 'MANUAL_FETCH:DOR_school_foundation', 'low',
            'Net foundation transfer to/from county after recapture. '
            'Positive = county receives guarantee funds; negative = county remits recapture. '
            'Source: WY DOR Annual Report school finance tables; '
            'Legislative Service Office school finance model.'
        ),
    }

assert len(fiscal_baseline) == 23, f'Expected 23 counties, got {len(fiscal_baseline)}'
print(f'Assembled fiscal_baseline for {len(fiscal_baseline)} WY counties.')
print(f'Sample keys for Campbell (56005):')
print(list(fiscal_baseline['56005'].keys()))

Assembled fiscal_baseline for 23 WY counties.
Sample keys for Campbell (56005):
['geoid', 'county_name', 'state', 'fiscal_schema_version', 'assessed_values', 'mill_levy_composite_mills', 'mineral_production_valuation', 'severance_tax_distribution_usd', 'federal_mineral_royalties_usd', 'pilt_usd', 'sales_use_tax_distribution_usd', 'datacenter_equipment_exemption', 'employment_bls_qcew', 'labor_force_laus', 'school_finance_foundation_net_usd']


In [9]:
# ── Sanity check: mineral share of total assessed value ───────────────────────────
#
# For each county: mineral_assessed / total_assessed × 100%.
# Flags:
#   Campbell (56005): mineral share should dominate — historically ≥80%. Flag if <60%.
#   Teton (56039):    residential-dominated — residential >50%. Flag if mineral >20%.
#   Others:           flag if class shares look implausible given known industry.
#
# Because DOR assessed values are currently all null (pending MANUAL_FETCH.md),
# this cell will print a 'DATA PENDING' notice. Re-run after manual entry.

print('=' * 72)
print('SANITY TABLE — Mineral share of total assessed value')
print('(All DOR values are null until MANUAL_FETCH.md is completed)')
print('=' * 72)

SANITY_ANCHORS = {
    '56005': {'check': 'mineral_dominant', 'threshold': 0.60,
               'description': 'Campbell: PRB coal — expect mineral ≥80% (flag if <60%)'},
    '56039': {'check': 'residential_dominant', 'threshold': 0.50,
               'description': 'Teton: resort/residential — expect residential >50%, mineral <20%'},
    '56037': {'check': 'mineral_and_industrial', 'threshold': 0.40,
               'description': 'Sweetwater: trona+Jim Bridger — expect mineral+industrial >40%'},
    '56021': {'check': 'mixed',
               'description': 'Laramie: mixed (commercial/residential/industrial from DC+UW)'},
    '56023': {'check': 'mixed',
               'description': 'Lincoln: Kemmerer/Natrium — expect industrial growing after 2032'},
}

header = f"{'County':<16} {'Mineral':>10} {'Industrial':>10} {'Residential':>12} {'Mineral%':>9} {'Status':>12}"
print(header)
print('-' * 72)

flags = []

for geoid, county_name in WY_COUNTIES.items():
    av = fiscal_baseline[geoid]['assessed_values']
    mineral = av['mineral']['value']
    industrial = av['industrial']['value']
    residential = av['residential']['value']
    commercial = av['commercial']['value']
    agricultural = av['agricultural']['value']
    all_other = av['all_other']['value']

    # All values from the same row — compute total only if all are present
    values_present = [v for v in [mineral, industrial, residential, commercial, agricultural, all_other]
                      if v is not None]

    if len(values_present) < 6:
        status = 'DATA PENDING'
        mineral_pct_str = '—'
        mineral_str = '—'
        industrial_str = '—'
        residential_str = '—'
    else:
        total = sum(values_present)
        mineral_pct = mineral / total if total > 0 else 0.0
        mineral_pct_str = f'{mineral_pct:.1%}'
        mineral_str = f'${mineral/1e6:.0f}M'
        industrial_str = f'${industrial/1e6:.0f}M'
        residential_str = f'${residential/1e6:.0f}M'

        anchor = SANITY_ANCHORS.get(geoid, {})
        check = anchor.get('check', '')
        threshold = anchor.get('threshold', None)

        if check == 'mineral_dominant' and mineral_pct < threshold:
            status = f'FLAG ⚠ (<{threshold:.0%})'
            flags.append((county_name, geoid, anchor.get('description', ''), f'mineral_pct={mineral_pct:.1%}'))
        elif check == 'residential_dominant':
            res_pct = residential / total if total > 0 else 0
            min_pct = mineral / total if total > 0 else 0
            if res_pct < threshold or min_pct > 0.20:
                status = f'FLAG ⚠'
                flags.append((county_name, geoid, anchor.get('description', ''),
                              f'res_pct={res_pct:.1%}, mineral_pct={min_pct:.1%}'))
            else:
                status = 'OK'
        else:
            status = 'OK'

    print(f"{county_name:<16} {mineral_str:>10} {industrial_str:>10} {residential_str:>12} {mineral_pct_str:>9} {status:>12}")

print('=' * 72)

if all(fiscal_baseline[g]['assessed_values']['mineral']['value'] is None for g in WY_COUNTIES):
    print()
    print('⚠ ALL ASSESSED VALUES ARE NULL — complete MANUAL_FETCH.md before finalising JSON.')
    print('  The sanity check cannot run until DOR Annual Report values are entered.')
    print()
    print('Expected patterns after data entry:')
    for geoid, anchor in SANITY_ANCHORS.items():
        print(f'  {WY_COUNTIES[geoid]} ({geoid}): {anchor["description"]}')
elif flags:
    print()
    print('FLAGGED COUNTIES — investigate before finalising:')
    for county_name, geoid, desc, detail in flags:
        print(f'  ⚠ {county_name} ({geoid}): {desc}')
        print(f'     Detail: {detail}')
        print(f'     → Add a markdown cell explaining the deviation before proceeding.')
else:
    print('All sanity checks passed.')

SANITY TABLE — Mineral share of total assessed value
(All DOR values are null until MANUAL_FETCH.md is completed)
County              Mineral Industrial  Residential  Mineral%       Status
------------------------------------------------------------------------
Albany                 $24M       $87M        $325M      3.5%           OK
Big Horn               $97M       $17M         $76M     37.5%           OK
Campbell             $3798M      $355M        $303M     78.5%           OK
Carbon                $170M      $179M        $116M     23.9%           OK
Converse             $3114M      $353M        $111M     81.5%           OK
Crook                  $81M       $17M         $84M     28.5%           OK
Fremont               $199M       $54M        $294M     28.6%           OK
Goshen                  $2M       $11M         $84M      0.8%           OK
Hot Springs            $99M        $7M         $36M     57.7%           OK
Johnson               $213M       $41M        $109M     50.3%  

In [10]:
# ── Write wy_county_fiscal_baseline.json ─────────────────────────────────────────
#
# Top-level structure:
#   state_severance_by_mineral  → statewide severance tax by mineral type (W2 input)
#   state_class_shares          → statewide assessed value class shares (W2 input)
#   counties                    → 23 WY county records (geoid-keyed)

output_json = {
    'state_severance_by_mineral': STATE_SEVERANCE_BY_MINERAL,
    'state_class_shares':         STATE_CLASS_SHARES,
    'counties':                   fiscal_baseline,
}

out_json = PROCESSED / 'wy_county_fiscal_baseline.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(output_json, f, indent=2, ensure_ascii=False)

print(f'Written: {out_json}')
print(f'Size: {out_json.stat().st_size:,} bytes')
print(f'Top-level keys: {list(output_json.keys())}')
print(f'Counties: {len(fiscal_baseline)}')

missing = set(WY_COUNTIES.keys()) - set(fiscal_baseline.keys())
if missing:
    print(f'MISSING COUNTIES: {missing}')
else:
    print('All 23 WY counties present [OK]')

missing_laus = [g for g in fiscal_baseline if 'labor_force_laus' not in fiscal_baseline[g]]
if missing_laus:
    print(f'WARNING: labor_force_laus key missing from: {missing_laus}')
else:
    print('labor_force_laus key present in all 23 counties [OK]')

bare = []
for geoid, rec in fiscal_baseline.items():
    for k, v in rec.items():
        if isinstance(v, (int, float)) and k not in ('geoid', 'county_name', 'state',
                                                       'fiscal_schema_version'):
            bare.append((geoid, k))
if bare:
    print(f'WARNING: bare numbers found: {bare[:5]}')
else:
    print('No bare numbers in county records [OK]')


Written: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/wy_county_fiscal_baseline.json
Size: 283,145 bytes
Top-level keys: ['state_severance_by_mineral', 'state_class_shares', 'counties']
Counties: 23
All 23 WY counties present [OK]
labor_force_laus key present in all 23 counties [OK]
No bare numbers in county records [OK]


In [11]:
# ── Write wy_fiscal_sources.csv ───────────────────────────────────────────────────
#
# Format mirrors material_coefficient_sources.csv:
# county_geoid, item, value, unit, coverage, primary_input,
# source, year, notes, confidence

SOURCES_HEADER = [
    'county_geoid', 'item', 'value', 'unit', 'coverage',
    'primary_input', 'source', 'year', 'notes', 'confidence'
]

def wrap_to_source_row(geoid: str, item: str, unit: str,
                        coverage: str, primary_input: str,
                        wrapped: Dict) -> Dict:
    return {
        'county_geoid': geoid,
        'item':         item,
        'value':        wrapped.get('value', None),
        'unit':         unit,
        'coverage':     coverage,
        'primary_input':primary_input,
        'source':       wrapped.get('source', ''),
        'year':         wrapped.get('year', ''),
        'notes':        wrapped.get('notes', ''),
        'confidence':   wrapped.get('confidence', ''),
    }


source_rows = []

for geoid, county_name in WY_COUNTIES.items():
    rec = fiscal_baseline[geoid]

    # Assessed values
    for cls in ['mineral', 'industrial', 'commercial', 'residential', 'agricultural', 'all_other']:
        source_rows.append(wrap_to_source_row(
            geoid, f'assessed_value_{cls}', 'USD (post-ratio assessed value)',
            'county total', 'fiscal_base',
            rec['assessed_values'][cls]
        ))

    # Mill levy
    source_rows.append(wrap_to_source_row(
        geoid, 'mill_levy_composite', 'mills',
        'county composite', 'fiscal_base',
        rec['mill_levy_composite_mills']
    ))

    # Mineral production valuation — all keys (total + pct + per-commodity)
    for key in ['total_all_commodities', 'pct_statewide', 'coal', 'oil', 'gas', 'trona']:
        if key in rec['mineral_production_valuation']:
            unit = 'fraction_of_statewide' if key == 'pct_statewide' else 'USD'
            source_rows.append(wrap_to_source_row(
                geoid, f'mineral_production_valuation_{key}', unit,
                'county total', 'fiscal_base',
                rec['mineral_production_valuation'][key]
            ))

    # Severance tax
    source_rows.append(wrap_to_source_row(
        geoid, 'severance_tax_distribution', 'USD/year',
        'county distribution', 'revenue',
        rec['severance_tax_distribution_usd']
    ))

    # Federal mineral royalties
    source_rows.append(wrap_to_source_row(
        geoid, 'federal_mineral_royalties', 'USD/year',
        'county disbursement', 'revenue',
        rec['federal_mineral_royalties_usd']
    ))

    # PILT
    source_rows.append(wrap_to_source_row(
        geoid, 'pilt_payment', 'USD/year',
        'county total', 'revenue',
        rec['pilt_usd']
    ))

    # Sales & use tax
    source_rows.append(wrap_to_source_row(
        geoid, 'sales_use_tax_distribution', 'USD/year',
        'county distribution', 'revenue',
        rec['sales_use_tax_distribution_usd']
    ))

    # QCEW employment
    for label, industry_data in rec['employment_bls_qcew']['naics_industries'].items():
        for metric in ['annual_avg_emplvl', 'avg_annual_pay']:
            if metric in industry_data:
                source_rows.append(wrap_to_source_row(
                    geoid, f'employment_{label}_{metric}',
                    'workers' if 'emplvl' in metric else 'USD/year',
                    'county × NAICS', 'employment',
                    industry_data[metric]
                ))

    # LAUS labor force
    source_rows.append(wrap_to_source_row(
        geoid, 'labor_force_laus', 'persons',
        'county total labor force', 'employment',
        rec['labor_force_laus']
    ))

    # School finance
    source_rows.append(wrap_to_source_row(
        geoid, 'school_finance_foundation_net', 'USD/year',
        'county net foundation transfer', 'revenue',
        rec['school_finance_foundation_net_usd']
    ))


out_csv = PROCESSED / 'wy_fiscal_sources.csv'
with open(out_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=SOURCES_HEADER)
    writer.writeheader()
    writer.writerows(source_rows)

print(f'Written: {out_csv}')
print(f'Rows: {len(source_rows):,}')
print(f'Counties: {len(set(r["county_geoid"] for r in source_rows))}')

# Show confidence distribution
conf_counts = {}
for r in source_rows:
    c = r['confidence']
    conf_counts[c] = conf_counts.get(c, 0) + 1
print(f'Confidence breakdown: {conf_counts}')

Written: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/wy_fiscal_sources.csv
Rows: 667
Counties: 23
Confidence breakdown: {'high': 370, 'low': 297}


In [12]:
# ── Write MANUAL_FETCH.md ─────────────────────────────────────────────────────────
#
# Status after CSV extraction run (2026-06-17):
#   RESOLVED      : pilt_payments (pdf_print_counties.cfm.csv)
#   UNRECOVERABLE : table of distributions by county (raster image in PDF)
#   OUTSTANDING   : ONRR county disbursements (Item 2)
#   OUTSTANDING   : DOR Mineral Valuation Report per-county (Item 1)
#   OUTSTANDING   : Per-county severance tax distributions (Item 3 — pro-rata est. in place)
#   OUTSTANDING   : School finance foundation net (Item 4)

MANUAL_FETCH_CONTENT = (
    "# MANUAL_FETCH.md \u2014 Wyoming Fiscal Baseline Pull\n\n"
    "Generated by `17_wy_fiscal_pull.ipynb`. Re-run Cell 13 to regenerate.\n\n"
    "**Data loaded from extracted CSVs (do NOT re-fetch):**\n"
    "- \u2713 Assessed values by class \u2014 `State Assessed Values.csv` + `Locally Assessed Values .csv`\n"
    "- \u2713 Total assessed (sanity denominator) \u2014 `Comparison of State and Local Assessed Values.csv` metric_6\n"
    "- \u2713 Composite mill levy (primary W2 input) \u2014 `Grand total all taxes levied.csv` col `_1`\n"
    "- \u2713 Mineral-weighted mill levy \u2014 `County and Statewide weighted average mill levies.csv`\n"
    "- \u2713 Mineral production valuation (total) \u2014 `Total mineral taxable value of production by county.csv`\n"
    "- \u2713 **PILT payments RESOLVED** \u2014 `pdf_print_counties.cfm.csv` (previously outstanding)\n"
    "- \u2713 Sales/use tax distribution \u2014 `Sales and use tax distribution report.csv` (grand_total)\n"
    "- \u2713 Statewide severance by mineral \u2014 `mineral severance tax distribution.csv` (top-level JSON)\n"
    "- \u2713 State class shares \u2014 `Percentage of local and state assessed values.csv` (top-level JSON)\n"
    "- \u2713 BLS QCEW employment \u2014 live API pull (23 counties, NAICS 2121/2111/2211/23/518210)\n"
    "- \u2713 BLS LAUS labor force \u2014 live API pull (23 counties, annual avg M01\u2013M12)\n"
    "- \u2713 Park (56029) county_sales_tax = $0 confirmed genuine (no county-option levy)\n"
    "- \u2713 Sublette (56035) county_sales_tax = $0 confirmed genuine (no county-option levy)\n\n"
    "**UNRECOVERABLE:**\n"
    "- \u2717 `table of distributions by county.csv` \u2014 **unrecoverable raster image**. "
    "PDF page is a scanned table with no machine-readable text. "
    "Cannot be extracted programmatically. "
    "If county-level lodging/distribution breakdown is needed, obtain directly from WY DOR.\n\n"
    "**Items still requiring manual fetch:**\n\n---\n\n"
    "## Item 1 \u2014 Per-County Mineral Production by Commodity (Coal / Oil / Gas / Trona)\n\n"
    "**Target:** `DOR_MINERAL_PROD[geoid][\"coal\" | \"oil\" | \"gas\" | \"trona\"]` in Cell 8\n"
    "**Data year:** 2025 (DOR 2025 Annual Report)\n"
    "**Current value:** null (confidence: low)\n\n"
    "**Source:** WY DOR Mineral Tax Division \u2014 Annual Mineral Valuation Report\n"
    "- URL: https://revenue.wyo.gov/mineral-tax-division\n"
    "- Look for \u201cMineral Valuation Summary by County\u201d or equivalent table\n"
    "- Columns needed per county: Coal, Oil, Natural Gas, Trona production valuation (USD)\n"
    "- Statewide totals available in `STATE_SEVERANCE_BY_MINERAL` (top-level JSON):\n"
    "  Coal (surface): ~$132M, Oil: ~$444M, Natural Gas: ~$149M, Trona: ~$38M\n\n"
    "**Sanity anchors:**\n"
    "- Campbell (56005): Coal dominates \u2014 PRB, largest US coal county. Expect coal >> $1B.\n"
    "- Sweetwater (56037): Trona significant + some gas. Coal near zero.\n"
    "- Sublette (56035): Gas-dominated (Pinedale Anticline). Coal \u2248 0.\n"
    "- Teton (56039): All commodities \u2248 0.\n\n"
    "**How to enter:** In Cell 8, replace the four `wrap(None, ..., 'MANUAL_FETCH:mineral_prod_*')`\n"
    "calls inside `DOR_MINERAL_PROD[geoid]` with `wrap(actual_value, DATA_YEAR, SRC_MINERAL, 'high')`.\n"
    "Re-run cells 8\u201313 after entry.\n\n---\n\n"
    "## Item 2 \u2014 County-Level ONRR Federal Mineral Royalties\n\n"
    "**Target:** `ONRR_FMR[geoid]` in Cell 8\n"
    "**Data year:** FY 2024 (most recent available)\n"
    "**Current value:** null (confidence: low)\n\n"
    "**Source:** ONRR Revenue Data (JavaScript-required)\n"
    "- URL: https://revenuedata.onrr.gov/downloads/disbursements/\n"
    "- Download \u201cCounty Disbursements\u201d CSV; filter State=Wyoming, Year=FY 2024\n"
    "- Sum all disbursement types (royalties, rents, bonuses, other) per county\n\n"
    "**Note:** Federal mineral royalties \u2260 severance tax. Collected by ONRR; "
    "50% to states under Mineral Leasing Act; WY share further distributed to counties by legislature. "
    "Separate ledger, different lag structure.\n\n---\n\n"
    "## Item 3 \u2014 Per-County Severance Tax Distribution\n\n"
    "**Target:** `DOR_SEVERANCE[geoid]` in Cell 8\n"
    "**Data year:** 2025\n"
    "**Current value:** Pro-rata estimate (county mineral share \u00d7 $774,028,998 statewide)\n\n"
    "**Source:** WY DOR Tax Distribution Reports\n"
    "- URL: https://revenue.wyo.gov/tax-distribution-reports\n"
    "- Navigate to \u201cMineral Tax Distributions\u201d\n"
    "- Download calendar year 2024 or 2025 report\n"
    "- Statewide total verified: $774,028,998\n\n"
    "**Key statutory formulas:** Coal: W.S. 39-14-801 | Oil & gas: W.S. 39-14-205 | Trona: W.S. 39-14-505\n\n---\n\n"
    "## Item 4 \u2014 School Finance Foundation Net Transfer\n\n"
    "**Target:** `school_finance_foundation_net_usd` in each county record (Cell 9)\n"
    "**Data year:** school FY 2024\u20132025\n"
    "**Current value:** null (confidence: low)\n\n"
    "**Source:**\n"
    "- WY Legislative Service Office (LSO): https://lso.wyoming.gov/school-finance\n"
    "- Or WY DOR Annual Report school finance tables\n"
    "- Net transfer = guarantee receipts \u2212 recapture remittances\n"
    "- Note: Campbell and Sweetwater likely remit recapture (high mineral AV)\n\n---\n\n"
    "## Post-Entry Protocol\n\n"
    "After populating any item above:\n"
    "1. Open `17_wy_fiscal_pull.ipynb`\n"
    "2. In Cell 8, replace the relevant `wrap(None, ...)` stubs with `wrap(actual_value, ...)`\n"
    "3. Re-run cells 8\u201313 to regenerate all output files\n"
    "4. Confirm sanity table still passes (Cell 10):\n"
    "   - Campbell (56005): mineral share \u2265 60% (currently 78.5%) \u2713\n"
    "   - Teton (56039): mineral share < 20% (currently 0.1%) \u2713, residential > 50% \u2713\n"
    "5. Confirm all 23 counties present and `labor_force_laus` populated in JSON\n"
)

out_md = PROCESSED / 'MANUAL_FETCH.md'
with open(out_md, 'w', encoding='utf-8') as f:
    f.write(MANUAL_FETCH_CONTENT)

print(f'Written: {out_md}')
print(f'Size: {out_md.stat().st_size:,} bytes')
print()
print('MANUAL_FETCH.md status:')
print('  RESOLVED      : pilt_payments (pdf_print_counties.cfm.csv)')
print('  UNRECOVERABLE : table of distributions by county (raster image)')
print('  OUTSTANDING   : ONRR county disbursements (Item 2)')
print('  OUTSTANDING   : DOR Mineral Valuation Report per-county (Item 1)')
print('  OUTSTANDING   : Severance tax county distributions (Item 3 -- pro-rata est. in place)')
print('  OUTSTANDING   : School finance foundation net (Item 4)')


Written: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/MANUAL_FETCH.md
Size: 5,163 bytes

MANUAL_FETCH.md status:
  RESOLVED      : pilt_payments (pdf_print_counties.cfm.csv)
  UNRECOVERABLE : table of distributions by county (raster image)
  OUTSTANDING   : ONRR county disbursements (Item 2)
  OUTSTANDING   : DOR Mineral Valuation Report per-county (Item 1)
  OUTSTANDING   : Severance tax county distributions (Item 3 -- pro-rata est. in place)
  OUTSTANDING   : School finance foundation net (Item 4)


## Handoff Conditions Checklist — NB17 WY Fiscal Baseline

| Condition | Status |
|-----------|--------|
| All 23 WY counties present in `wy_county_fiscal_baseline.json` | ✓ |
| Every value sourced or flagged (no bare numbers) | ✓ |
| `labor_force_laus` key present in all 23 records (W4 denominator dependency) | ✓ |
| BLS QCEW employment (NAICS 2121/2111/2211/23/518210) | ✓ Live pull |
| BLS LAUS labor force (annual avg M01–M12) | ✓ Live pull |
| Assessed values — mineral, all_other, industrial, commercial, residential, agricultural | ✓ Loaded from CSVs |
| Composite mill levy — `Grand total all taxes levied.csv` col `_1` | ✓ Loaded |
| Mineral-weighted mill levy — `County and Statewide weighted average mill levies.csv` | ✓ Loaded |
| Mineral production — total all commodities | ✓ Loaded |
| Mineral production — coal/oil/gas/trona per county | ⚠ null; needs DOR Mineral Valuation Report (Item 1) |
| Severance tax distributions | ⚠ Pro-rata estimate (confidence: low); see Item 3 |
| Federal mineral royalties (ONRR) | ⚠ null per county; see Item 2 |
| PILT payments | ✓ Loaded — `pdf_print_counties.cfm.csv` (RESOLVED) |
| Sales/use tax distributions | ✓ `grand_total` loaded; Park + Sublette $0 county-option confirmed genuine |
| `state_severance_by_mineral` at top level of JSON | ✓ |
| `state_class_shares` at top level of JSON | ✓ |
| `table of distributions by county.csv` | ✗ UNRECOVERABLE — raster image in PDF |
| School finance foundation net | ⚠ null; see Item 4 |
| Sanity check: Campbell mineral share ≥ 60% | ✓ 78.5% |
| Sanity check: Teton mineral share < 20% | ✓ 0.1% |
| All 23 counties in sanity table with no bare numbers | ✓ |
| `datacenter_equipment_exemption` structural field present | ✓ |

**Outstanding for W2:** Items 1–4 in MANUAL_FETCH.md.
Per-county coal/oil/gas/trona valuations (Item 1) and ONRR county disbursements (Item 2)
are the highest-impact gaps for the fiscal coefficient derivation.
